# AdireTrust — DINOv2 linear probe (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drizzy765/adiretrust-app/blob/main/model/train_colab.ipynb)

All pretraining for AdireTrust runs in this notebook on a **Colab GPU**. It trains a **frozen DINOv2 ViT-S/14** backbone plus a **logistic regression** probe to distinguish hand-dyed Abeokuta adire from factory prints.

**What this notebook does**

1. Check that a GPU is available.
2. Clone [`drizzy765/adiretrust-app`](https://github.com/drizzy765/adiretrust-app) and install extra packages (torch is already on Colab).
3. Optionally mount Google Drive if you have the missing Osun **print** photos.
4. Load `data/labels/labels.csv` and `splits.csv`. Synthetics stay in **train only**.
5. Resize to 224×224, ImageNet-normalize, extract frozen features, cache `.npy` files.
6. Train the probe, evaluate on val/test with **metrics computed in this run** (nothing is hardcoded), and analyze an abstain threshold of **0.6**.
7. Save `model/weights/*.joblib` plus a label map, then optionally copy them to Drive or download them.

**Honest-reporting rules**

- Do not invent accuracy, paste a number from a paper, or treat a stub API response as a model score.
- A false “genuine” on a factory print is the costly error. Prefer abstaining over a confident wrong genuine call.
- Current Osun IDs are provisional (one image ≈ one piece). Do not call a split metric a physical-piece held-out benchmark until views of the same cloth are grouped.
- The public repo includes **81 genuine** Osun photos and **24 train-only synthetics**. The manifest lists **100 print** photos that were not in the public git history. If those files are still missing, val/test may contain only genuine images — say that out loud in `EVAL.md` and do not present the run as a finished AdireTrust result.
- Write every reported number from the cells below into `EVAL.md` with **n**, the split rule, and the date.


## 1. Runtime and GPU check

In Colab: **Runtime → Change runtime type → GPU** (T4 is enough). Then run this cell.


In [ ]:
import sys
import torch

print("Python", sys.version.split()[0])
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. Feature extraction will be slow on CPU. "
        "Use Runtime → Change runtime type → GPU, then rerun this cell."
    )
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Clone the repo and install packages

Torch is preinstalled on Colab. We add `timm` (optional), scikit-learn, pandas, Pillow, and joblib. Features are extracted with `torch.hub` (`facebookresearch/dinov2`, `dinov2_vits14`).


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/drizzy765/adiretrust-app"
REPO_NAME = "adiretrust-app"


def has_manifest(root: Path) -> bool:
    return (root / "data" / "labels" / "labels.csv").is_file()


cwd = Path.cwd().resolve()
if has_manifest(cwd):
    ROOT = cwd
elif has_manifest(cwd / REPO_NAME):
    ROOT = cwd / REPO_NAME
else:
    if not (cwd / REPO_NAME).exists():
        subprocess.check_call(["git", "clone", REPO_URL])
    ROOT = cwd / REPO_NAME

os.chdir(ROOT)
print("Repo root:", ROOT)

# timm is optional; the extractor below uses torch.hub DINOv2.
%pip -q install scikit-learn pandas pillow joblib timm


## 3. Optional: mount the missing print photos from Google Drive

The public git repo does **not** include `data/raw/osun_prints/`. `labels.csv` lists 100 Osun workshop print images, but those files were never in the public tree (the research repo gitignored `/data/raw/`).

- If you have the consented print folder, upload it to Drive and set `MOUNT_PRINTS = True` plus the folder path.
- **Do not** invent, scrape, or substitute random Ankara photos to fill the gap.
- If you skip this cell, the notebook will still run, but it will warn when print files are missing and when val/test lack one class.


In [ ]:
from pathlib import Path
import shutil

MOUNT_PRINTS = False  # set True only if you have the consented print folder
DRIVE_PRINTS = Path("/content/drive/MyDrive/adiretrust/osun_prints")  # change to your path
DEST_PRINTS = ROOT / "data" / "raw" / "osun_prints"

if MOUNT_PRINTS:
    from google.colab import drive

    drive.mount("/content/drive")
    if not DRIVE_PRINTS.exists():
        raise FileNotFoundError(
            f"No print folder at {DRIVE_PRINTS}. Fix DRIVE_PRINTS or set MOUNT_PRINTS = False."
        )
    DEST_PRINTS.mkdir(parents=True, exist_ok=True)
    copied = 0
    for src in DRIVE_PRINTS.rglob("*"):
        if src.is_file() and src.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            target = DEST_PRINTS / src.name
            if not target.exists():
                shutil.copy2(src, target)
            copied += 1
    print(f"Print images available in {DEST_PRINTS}: {copied}")
else:
    n = len(list(DEST_PRINTS.glob("*"))) if DEST_PRINTS.exists() else 0
    print(
        "Drive mount skipped. Print files already in the repo folder:"
        if n
        else "Drive mount skipped. data/raw/osun_prints/ is empty or missing."
    )
    print("Count:", n)


## 4. Load labels and splits

Rules used here:

- Honor `splits.csv` (`train` / `val` / `test`).
- `label_source == synthetic` may appear in **train only**. Never in val/test, and never in a reported test score.
- Skip `wikimedia_commons` rows (unsplit, not copied into this repo).
- Skip `uncertain` labels for the supervised probe.
- Prefer existing 224px workshop crops when present; otherwise load the raw file from the path in the CSV.


In [ ]:
from collections import Counter
from pathlib import Path
import csv

import pandas as pd

LABELS_PATH = ROOT / "data" / "labels" / "labels.csv"
SPLITS_PATH = ROOT / "data" / "labels" / "splits.csv"
LABELS = ("genuine_hand_dyed", "factory_print")

labels_df = pd.read_csv(LABELS_PATH)
splits_df = pd.read_csv(SPLITS_PATH)
split_of = dict(zip(splits_df["piece_id"].astype(str), splits_df["split"].astype(str)))


def resolve_image(row) -> Path | None:
    relative = Path(str(row["filename"]).replace("\\", "/"))
    candidates = [
        ROOT / relative,
        ROOT / "data" / "processed" / "osun_workshop" / "224" / relative.name,
        ROOT / "data" / "raw" / "osun_genuine" / relative.name,
        ROOT / "data" / "raw" / "osun_prints" / relative.name,
        ROOT / "data" / "raw" / "synthetic" / relative.name,
    ]
    for path in candidates:
        if path.is_file():
            return path
    return None


rows = []
skipped = Counter()
for row in labels_df.to_dict(orient="records"):
    source = str(row.get("label_source") or "")
    if source == "wikimedia_commons":
        skipped["commons"] += 1
        continue
    split = split_of.get(str(row["piece_id"]))
    if not split:
        skipped["unsplit"] += 1
        continue
    if source == "synthetic" and split != "train":
        skipped["synthetic_not_train"] += 1
        continue
    if row.get("label") not in LABELS:
        skipped["uncertain_or_other"] += 1
        continue
    path = resolve_image(row)
    if path is None:
        skipped["missing_file"] += 1
        continue
    rows.append(
        {
            "piece_id": str(row["piece_id"]),
            "label": row["label"],
            "split": split,
            "source": source,
            "path": str(path),
        }
    )

examples = pd.DataFrame(rows)
print("Usable labeled images:", len(examples))
print("By split:", examples["split"].value_counts().to_dict() if len(examples) else {})
print("By label:", examples["label"].value_counts().to_dict() if len(examples) else {})
print("By source:", examples["source"].value_counts().to_dict() if len(examples) else {})
print("Skipped:", dict(skipped))
if skipped.get("missing_file"):
    print(
        f"WARNING: {skipped['missing_file']} manifest rows have no image file. "
        "If these are the Osun prints, mount them from Drive in the previous cell. "
        "Do not invent a print set."
    )

for part in ("train", "val", "test"):
    subset = examples[examples["split"] == part]
    print(f"\n{part}: {len(subset)}  labels={subset['label'].value_counts().to_dict()}")


## 5. Preprocess (224×224, ImageNet normalize) and extract frozen DINOv2 features

Backbone: `torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")` — **frozen**, eval mode, no fine-tuning.

Features are cached under `model/runs/` as `.npy` files so you can rerun the probe without extracting again. Those caches are gitignored (they can be large).


In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGE_SIZE = 224
BATCH_SIZE = 32
CACHE_DIR = ROOT / "model" / "runs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

preprocess = transforms.Compose(
    [
        transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)


class ClothDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        return preprocess(image), row["label"], row["split"], row["piece_id"]


def extract_features(frame: pd.DataFrame, model, device) -> dict:
    loader = DataLoader(ClothDataset(frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    feats, labels, splits, ids = [], [], [], []
    model.eval()
    with torch.no_grad():
        for batch, batch_labels, batch_splits, batch_ids in loader:
            batch = batch.to(device, non_blocking=True)
            out = model(batch)
            if isinstance(out, (tuple, list)):
                out = out[0]
            feats.append(out.detach().cpu().numpy())
            labels.extend(list(batch_labels))
            splits.extend(list(batch_splits))
            ids.extend(list(batch_ids))
    return {
        "features": np.concatenate(feats, axis=0) if feats else np.zeros((0, 384), dtype=np.float32),
        "labels": np.array(labels),
        "splits": np.array(splits),
        "piece_ids": np.array(ids),
    }


CACHE_PATH = CACHE_DIR / "dinov2_vits14_features.npz"
if CACHE_PATH.is_file():
    packed = dict(np.load(CACHE_PATH, allow_pickle=True))
    print("Loaded cached features:", CACHE_PATH, packed["features"].shape)
else:
    if examples.empty:
        raise SystemExit("No usable images were found. Check the data cells above.")
    print("Loading frozen DINOv2 ViT-S/14 from torch.hub (facebookresearch/dinov2, dinov2_vits14)…")
    backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
    backbone.to(DEVICE)
    for param in backbone.parameters():
        param.requires_grad = False
    backbone.eval()
    packed = extract_features(examples, backbone, DEVICE)
    np.savez_compressed(CACHE_PATH, **packed)
    print("Cached features:", CACHE_PATH, packed["features"].shape)
    del backbone
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 6. Train a logistic regression probe

The vision backbone stays frozen. Only this small classifier is fit, and only on the **train** split.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

X = packed["features"]
y_raw = packed["labels"]
splits = packed["splits"]

train_mask = splits == "train"
val_mask = splits == "val"
test_mask = splits == "test"

X_train, y_train_raw = X[train_mask], y_raw[train_mask]
X_val, y_val_raw = X[val_mask], y_raw[val_mask]
X_test, y_test_raw = X[test_mask], y_raw[test_mask]

print("Train", X_train.shape, Counter(y_train_raw))
print("Val  ", X_val.shape, Counter(y_val_raw))
print("Test ", X_test.shape, Counter(y_test_raw))

train_classes = set(y_train_raw.tolist())
if train_classes != set(LABELS):
    raise SystemExit(
        "Training needs both genuine_hand_dyed and factory_print examples. "
        f"Found {sorted(train_classes)}. If print photos are missing, mount them "
        "from Drive and rerun from the data cells. Do not invent scores."
    )

encoder = LabelEncoder()
encoder.fit(list(LABELS))
y_train = encoder.transform(y_train_raw)
y_val = encoder.transform(y_val_raw) if len(y_val_raw) else np.array([], dtype=int)
y_test = encoder.transform(y_test_raw) if len(y_test_raw) else np.array([], dtype=int)

probe = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    C=1.0,
)
probe.fit(X_train, y_train)
print("Classes:", list(encoder.classes_))
print("Fitted logistic regression probe.")


## 7. Evaluate on val and test

Every number below is computed from this run. If a split is missing a class, sklearn may warn and some metrics will be undefined or one-sided — record that, do not fill in a fake score.

Precision on **genuine** is the headline metric (a false genuine on a print is the costly error).


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)

GENUINE = "genuine_hand_dyed"


def evaluate_split(name, features, y_true_raw):
    print("=" * 60)
    print(name, "n =", len(y_true_raw), "label counts:", dict(Counter(y_true_raw)))
    if len(y_true_raw) == 0:
        print("No rows in this split.")
        return None
    proba = probe.predict_proba(features)
    y_pred = encoder.inverse_transform(probe.predict(features))
    labels_present = list(encoder.classes_)
    acc = accuracy_score(y_true_raw, y_pred)
    prec_genuine = precision_score(
        y_true_raw,
        y_pred,
        labels=labels_present,
        pos_label=GENUINE,
        zero_division=0,
    )
    rec_genuine = recall_score(
        y_true_raw,
        y_pred,
        labels=labels_present,
        pos_label=GENUINE,
        zero_division=0,
    )
    cm = confusion_matrix(y_true_raw, y_pred, labels=labels_present)
    print(f"accuracy = {acc:.4f}")
    print(f"precision_on_genuine = {prec_genuine:.4f}")
    print(f"recall_on_genuine = {rec_genuine:.4f}")
    print("confusion matrix rows=true, cols=pred, order:", labels_present)
    print(cm)
    print(classification_report(y_true_raw, y_pred, labels=labels_present, zero_division=0))
    if len(set(y_true_raw.tolist())) < 2:
        print(
            "WARNING: this split does not contain both classes. "
            "Do not treat these numbers as a complete AdireTrust benchmark."
        )
    return {
        "split": name,
        "n": int(len(y_true_raw)),
        "label_counts": {k: int(v) for k, v in Counter(y_true_raw).items()},
        "accuracy": float(acc),
        "precision_on_genuine": float(prec_genuine),
        "recall_on_genuine": float(rec_genuine),
        "confusion_matrix": cm.tolist(),
        "label_order": labels_present,
        "y_true": y_true_raw,
        "y_pred": y_pred,
        "proba": proba,
    }


val_metrics = evaluate_split("val", X_val, y_val_raw)
test_metrics = evaluate_split("test", X_test, y_test_raw)


## 8. Abstain threshold (0.6)

If the probe’s max class probability is below `CONFIDENCE_THRESHOLD` (default **0.6**), the call is `uncertain` instead of genuine or print. Tune this idea on **val**, then apply the same threshold to **test**. Do not pick a threshold by peeking at test.


In [ ]:
CONFIDENCE_THRESHOLD = 0.6


def apply_abstain(y_true_raw, proba, threshold=CONFIDENCE_THRESHOLD):
    pred_idx = proba.argmax(axis=1)
    conf = proba.max(axis=1)
    y_pred = encoder.inverse_transform(pred_idx)
    y_pred = np.where(conf >= threshold, y_pred, "uncertain")
    covered = conf >= threshold
    n = len(y_true_raw)
    n_cover = int(covered.sum())
    n_abstain = n - n_cover
    result = {
        "n": n,
        "n_covered": n_cover,
        "n_abstained": n_abstain,
        "coverage": float(n_cover / n) if n else 0.0,
        "abstain_rate": float(n_abstain / n) if n else 0.0,
        "y_pred": y_pred,
        "confidence": conf,
    }
    if n_cover:
        result["accuracy_covered"] = float(accuracy_score(y_true_raw[covered], y_pred[covered]))
        result["precision_on_genuine_covered"] = float(
            precision_score(
                y_true_raw[covered],
                y_pred[covered],
                labels=list(LABELS),
                pos_label=GENUINE,
                zero_division=0,
            )
        )
        # False genuine on a print, among covered calls
        false_genuine = int(((y_true_raw[covered] == "factory_print") & (y_pred[covered] == GENUINE)).sum())
        result["false_genuine_covered"] = false_genuine
    else:
        result["accuracy_covered"] = None
        result["precision_on_genuine_covered"] = None
        result["false_genuine_covered"] = None
    return result


def print_abstain(name, y_true_raw, proba):
    print("=" * 60)
    print(f"{name}  threshold={CONFIDENCE_THRESHOLD}")
    if len(y_true_raw) == 0:
        print("No rows.")
        return None
    stats = apply_abstain(y_true_raw, proba)
    print(
        f"n={stats['n']}  covered={stats['n_covered']}  abstained={stats['n_abstained']}  "
        f"coverage={stats['coverage']:.4f}  abstain_rate={stats['abstain_rate']:.4f}"
    )
    print("accuracy_covered:", stats["accuracy_covered"])
    print("precision_on_genuine_covered:", stats["precision_on_genuine_covered"])
    print("false_genuine_covered:", stats["false_genuine_covered"])
    return stats


val_abstain = print_abstain("val", y_val_raw, val_metrics["proba"] if val_metrics else np.zeros((0, 2)))
test_abstain = print_abstain("test", y_test_raw, test_metrics["proba"] if test_metrics else np.zeros((0, 2)))


## 9. Save the probe and copy / download it

Artifacts go to `model/weights/` (small `.joblib` / `.json` files; these are **not** gitignored so they can be committed):

- `dinov2_vits14_logreg.joblib` — the fitted sklearn probe
- `label_map.json` — class order and training metadata (no made-up scores)


In [ ]:
import json
from datetime import datetime, timezone

import joblib

WEIGHTS = ROOT / "model" / "weights"
WEIGHTS.mkdir(parents=True, exist_ok=True)

probe_path = WEIGHTS / "dinov2_vits14_logreg.joblib"
label_map_path = WEIGHTS / "label_map.json"

joblib.dump(
    {
        "probe": probe,
        "label_encoder_classes": list(encoder.classes_),
        "backbone": "facebookresearch/dinov2:dinov2_vits14",
        "image_size": IMAGE_SIZE,
        "normalize_mean": IMAGENET_MEAN,
        "normalize_std": IMAGENET_STD,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
    },
    probe_path,
)

label_map = {
    "classes": list(encoder.classes_),
    "index_to_label": {int(i): label for i, label in enumerate(encoder.classes_)},
    "label_to_index": {label: int(i) for i, label in enumerate(encoder.classes_)},
    "backbone": "facebookresearch/dinov2:dinov2_vits14",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "trained_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "n_train": int(len(y_train_raw)),
    "train_label_counts": {k: int(v) for k, v in Counter(y_train_raw).items()},
}
label_map_path.write_text(json.dumps(label_map, indent=2) + "\n", encoding="utf-8")
print("Wrote", probe_path, "size", probe_path.stat().st_size, "bytes")
print("Wrote", label_map_path)


In [ ]:
# Optional: copy to Drive and/or download in the browser.
COPY_TO_DRIVE = False
DRIVE_WEIGHTS = Path("/content/drive/MyDrive/adiretrust/weights")

if COPY_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_WEIGHTS.mkdir(parents=True, exist_ok=True)
    shutil.copy2(probe_path, DRIVE_WEIGHTS / probe_path.name)
    shutil.copy2(label_map_path, DRIVE_WEIGHTS / label_map_path.name)
    print("Copied weights to", DRIVE_WEIGHTS)

try:
    from google.colab import files

    files.download(str(probe_path))
    files.download(str(label_map_path))
except Exception as exc:
    print("Browser download skipped (not in Colab, or downloads blocked):", exc)
    print("Copy the files out of", WEIGHTS, "manually.")


## 10. Commit weights or hand them to the backend

After you have a real run (ideally with both classes in val/test):

```bash
# on your machine, after downloading the two files into model/weights/
git add model/weights/dinov2_vits14_logreg.joblib model/weights/label_map.json EVAL.md
git commit -m "Add DINOv2 linear probe weights from Colab"
git push origin main
```

Or upload the same two files for David to load in `POST /check`. The API must keep its stub behavior until those files are actually wired and tested. Do not check in `.npy` feature caches or raw torch checkpoints unless you mean to; `model/runs/` is gitignored.

The Docker API image still should not install torch unless inference is deployed and documented. Loading a joblib probe plus precomputed features is a later backend task; live DINOv2 inference on Render’s free tier may be tight, so a Colab-trained probe with a clearly documented runtime is the intended path.


## 11. Write `EVAL.md` from this run

The next cell writes `EVAL.md` using the **computed** dictionaries above. Read it before you commit. If print photos were missing, the file will say so — do not delete that warning to make the result look stronger.


In [ ]:
from datetime import datetime, timezone


def fmt_metrics(block):
    if not block:
        return "No rows in this split.\n"
    lines = [
        f"- n: {block['n']}",
        f"- label counts: {block['label_counts']}",
        f"- accuracy: {block['accuracy']:.4f}",
        f"- precision_on_genuine: {block['precision_on_genuine']:.4f}",
        f"- recall_on_genuine: {block['recall_on_genuine']:.4f}",
        f"- confusion matrix (rows=true, cols=pred, order={block['label_order']}): {block['confusion_matrix']}",
    ]
    if len(block["label_counts"]) < 2:
        lines.append("- warning: this split does not contain both classes; not a complete benchmark.")
    return "\n".join(lines) + "\n"


def fmt_abstain(block):
    if not block:
        return "No rows.\n"
    return (
        f"- n: {block['n']}\n"
        f"- covered: {block['n_covered']}\n"
        f"- abstained: {block['n_abstained']}\n"
        f"- coverage: {block['coverage']:.4f}\n"
        f"- abstain_rate: {block['abstain_rate']:.4f}\n"
        f"- accuracy_covered: {block['accuracy_covered']}\n"
        f"- precision_on_genuine_covered: {block['precision_on_genuine_covered']}\n"
        f"- false_genuine_covered: {block['false_genuine_covered']}\n"
    )


eval_path = ROOT / "EVAL.md"
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
missing_prints = int(skipped.get("missing_file", 0))
body = f"""# AdireTrust probe evaluation

Date (UTC): {now}

This file was generated by `model/train_colab.ipynb`. Every metric is from that run. Do not replace these figures with invented or paper-derived numbers.

## Setup

- Backbone: frozen DINOv2 ViT-S/14 (`torch.hub`, `facebookresearch/dinov2`, `dinov2_vits14`)
- Probe: sklearn `LogisticRegression` (balanced class weight)
- Input: 224×224, ImageNet mean/std
- Confidence / abstain threshold: {CONFIDENCE_THRESHOLD} (chosen in advance, inspected on val, applied to test)
- Split rule: `data/labels/splits.csv` (provisional piece IDs; one image per ID in the current Osun set). Synthetics restricted to train. Commons excluded.

## Data used

- Usable labeled images: {len(examples)}
- By split: {examples["split"].value_counts().to_dict() if len(examples) else {}}
- By label: {examples["label"].value_counts().to_dict() if len(examples) else {}}
- By source: {examples["source"].value_counts().to_dict() if len(examples) else {}}
- Skipped manifest rows: {dict(skipped)}
- Missing image files (often the 100 Osun prints not in the public repo): {missing_prints}

## Validation

{fmt_metrics(val_metrics)}
### Val at abstain threshold {CONFIDENCE_THRESHOLD}

{fmt_abstain(val_abstain)}
## Test

{fmt_metrics(test_metrics)}
### Test at abstain threshold {CONFIDENCE_THRESHOLD}

{fmt_abstain(test_abstain)}
## Limitations (keep these in the pitch)

- A photo is not a manufacturing certificate.
- If val or test lack factory-print images, do not present this as a finished genuine-vs-print result.
- Provisional piece IDs can leak the same cloth across crops if they are later found to be the same physical textile.
- Synthetic textures must not appear in val/test scores (they were train-only in this run).
"""
eval_path.write_text(body, encoding="utf-8")
print("Wrote", eval_path)
print(body)
